In [14]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [ ]:
result_path = Path("../results/downstream_task")
bias_types = ["less_positive_class"]
method_name_replacer = {"uniform": "Uniform", 
                        "kmm": "KMM",
                        "psa": "PSA",
                        "soft-mrs-linear": "Soft-MRS",
                        "mrs-forest": "MRS",
                        "fw-mrs-temperature": "FW-MRS",
                        "fw-mrs-temperature-svm": "FW-MRS$_{SVM}$",
                        }
data_set_replacer = {"diabetes": "Diabetes",
                    "folktables_employment": "Employment", 
                     "folktables_income": "Income",
                     "bank_marketing": "Bank Marketing",
                     "hr_analytics": "HR Analytic",
                     "german_credit": "German Credit", 
                     "breast_cancer": "Breast Cancer", 
                     "loan_prediction": "Loan",
                    }

In [ ]:
aurocs = []
auprcs = []
mccs = []
dict_list = []
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
        for method in method_name_replacer.keys():
            json_file = result_path / dataset / bias_type /  "0.1"/ method / "classification_results.json"
            try:
                result_file = pd.read_json(str(json_file))
            except FileNotFoundError:
                continue
            dict_list.append(
                {
                    "Method": method, "Data Set": dataset, 
                    "AUROC Mean": result_file["random forest auroc"]["mean"], 
                    "AUROC Std": result_file["random forest auroc"]["sd"], 
                    "MCC Mean": result_file["random forest mcc"]["mean"], 
                    "MCC Std": result_file["random forest mcc"]["sd"], 
                    "AUPRC Mean": result_file["random forest auprc"]["mean"], 
                    "AUPRC Std": result_file["random forest auprc"]["sd"], 
                    "Bias Type": bias_type, "Bias Strength": 0.1,
                    "Dropped Samples Mean": result_file["dropped_samples"]["mean"],
                    "Dropped Samples Std": result_file["dropped_samples"]["std"],
                }
                            )
result_df = pd.DataFrame(data=dict_list)

In [17]:
result_df = result_df.replace(method_name_replacer)
result_df

,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std
0,Uniform,diabetes,0.791498,0.022072,0.371281,0.037717,less_positive_class,0.1,0.00,0.000000
1,KMM,diabetes,0.781169,0.021943,0.357066,0.041765,less_positive_class,0.1,0.00,0.000000
2,PSA,diabetes,0.787277,0.023148,0.367046,0.037046,less_positive_class,0.1,0.00,0.000000
3,Soft-MRS,diabetes,0.787917,0.021554,0.367637,0.037645,less_positive_class,0.1,0.00,0.000000
4,MRS,diabetes,0.788607,0.023040,0.367308,0.039618,less_positive_class,0.1,49.90,32.116818
5,FW-MRS,diabetes,0.783521,0.024444,0.359864,0.039249,less_positive_class,0.1,45.20,33.881558
6,FW-MRS$_{SVM}$,diabetes,0.781510,0.029975,0.355858,0.047168,less_positive_class,0.1,82.70,34.441400
7,Uniform,folktables_employment,0.870601,0.010491,0.826992,0.017307,less_positive_class,0.1,0.00,0.000000
8,KMM,folktables_employment,0.856631,0.013580,0.808776,0.021985,less_positive_class,0.1,0.00,0.000000
9,PSA,folktables_employment,0.867401,0.010937,0.823689,0.017363,less_positive_class,0.1,0.04,0.280000


In [18]:
for bias_type in bias_types:
    for dataset in data_set_replacer.keys():
        mean_auroc_values = []
        std_auroc_values = []
        for method in result_df["Method"].unique():
            try:
                mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==0.1) & 
                                                    (result_df["Data Set"]==dataset)]["AUROC Mean"].iloc[0]
                mean_auroc_values.append(np.round(mean_auroc, 3))

                std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==0.1) & 
                                                    (result_df["Data Set"]==dataset)]["AUROC Std"].iloc[0]
                std_auroc_values.append(np.round(std_auroc, 2))
            except IndexError:
                mean_auroc_values.append(0)
                std_auroc_values.append(0)

        print(f"{data_set_replacer[dataset]} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ \
& ${mean_auroc_values[5]}\\pm{std_auroc_values[5]}$ \
& ${mean_auroc_values[6]}\\pm{std_auroc_values[6]}$ & \
\\\\")
    print("\n")

Diabetes & $0.791\pm0.02$ & $0.781\pm0.02$ & $0.787\pm0.02$ & $0.788\pm0.02$ & $0.789\pm0.02$ & $0.784\pm0.02$ & $0.782\pm0.03$ & \\
Employment & $0.871\pm0.01$ & $0.857\pm0.01$ & $0.867\pm0.01$ & $0.862\pm0.01$ & $0.87\pm0.01$ & $0.868\pm0.01$ & $0.864\pm0.01$ & \\
Income & $0.838\pm0.01$ & $0.82\pm0.01$ & $0.831\pm0.01$ & $0.827\pm0.01$ & $0.837\pm0.01$ & $0.836\pm0.01$ & $0.833\pm0.01$ & \\
Bank Marketing & $0.847\pm0.02$ & $0.832\pm0.03$ & $0.839\pm0.03$ & $0.834\pm0.03$ & $0.845\pm0.02$ & $0.843\pm0.02$ & $0.84\pm0.02$ & \\
HR Analytic & $0.753\pm0.02$ & $0.749\pm0.02$ & $0.75\pm0.02$ & $0.751\pm0.02$ & $0.751\pm0.02$ & $0.75\pm0.02$ & $0.751\pm0.02$ & \\
German Credit & $0.667\pm0.05$ & $0.649\pm0.05$ & $0.659\pm0.06$ & $0.657\pm0.05$ & $0.672\pm0.05$ & $0.642\pm0.06$ & $0.639\pm0.07$ & \\
Breast Cancer & $0.988\pm0.01$ & $0.989\pm0.01$ & $0.988\pm0.01$ & $0.989\pm0.01$ & $0.989\pm0.01$ & $0.981\pm0.01$ & $0.977\pm0.01$ & \\
Loan & $0.658\pm0.08$ & $0.61\pm0.1$ & $0.628\pm0.1$ & 

In [23]:
for bias_type in bias_types:
    print(f"{bias_type}, {0.1}")
    for dataset in data_set_replacer.keys():
        mean_auprc_values = []
        std_auprc_values = []
        for method in result_df["Method"].unique():
            try:
                mean_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==0.1) & 
                                                    (result_df["Data Set"]==dataset)]["AUPRC Mean"].iloc[0]
                mean_auprc_values.append(np.round(mean_auprc, 3))

                std_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==0.1) & 
                                                    (result_df["Data Set"]==dataset)]["AUPRC Std"].iloc[0]
                std_auprc_values.append(np.round(std_auprc, 2))
            except IndexError:
                mean_auprc_values.append(0)
                std_auprc_values.append(0)

        print(f"{data_set_replacer[dataset]} \
& ${mean_auprc_values[0]}\\pm{std_auprc_values[0]}$ \
& ${mean_auprc_values[1]}\\pm{std_auprc_values[1]}$ \
& ${mean_auprc_values[2]}\\pm{std_auprc_values[2]}$ \
& ${mean_auprc_values[3]}\\pm{std_auprc_values[3]}$ \
& ${mean_auprc_values[4]}\\pm{std_auprc_values[4]}$ \
& ${mean_auprc_values[5]}\\pm{std_auprc_values[5]}$ \
& ${mean_auprc_values[6]}\\pm{std_auprc_values[6]}$  \
& \\\\")
print("\n")

less_positive_class, 0.1
Diabetes & $0.371\pm0.04$ & $0.357\pm0.04$ & $0.367\pm0.04$ & $0.368\pm0.04$ & $0.367\pm0.04$ & $0.36\pm0.04$ & $0.356\pm0.05$  & \\
Employment & $0.827\pm0.02$ & $0.809\pm0.02$ & $0.824\pm0.02$ & $0.818\pm0.02$ & $0.826\pm0.02$ & $0.824\pm0.02$ & $0.819\pm0.02$  & \\
Income & $0.789\pm0.02$ & $0.765\pm0.02$ & $0.781\pm0.02$ & $0.774\pm0.02$ & $0.788\pm0.02$ & $0.787\pm0.02$ & $0.784\pm0.02$  & \\
Bank Marketing & $0.466\pm0.05$ & $0.446\pm0.06$ & $0.455\pm0.05$ & $0.447\pm0.05$ & $0.468\pm0.05$ & $0.463\pm0.05$ & $0.458\pm0.05$  & \\
HR Analytic & $0.455\pm0.03$ & $0.449\pm0.03$ & $0.454\pm0.03$ & $0.454\pm0.04$ & $0.456\pm0.03$ & $0.45\pm0.04$ & $0.454\pm0.03$  & \\
German Credit & $0.461\pm0.06$ & $0.443\pm0.06$ & $0.452\pm0.06$ & $0.451\pm0.06$ & $0.464\pm0.07$ & $0.436\pm0.06$ & $0.436\pm0.07$  & \\
Breast Cancer & $0.994\pm0.0$ & $0.995\pm0.0$ & $0.994\pm0.0$ & $0.994\pm0.0$ & $0.995\pm0.0$ & $0.989\pm0.01$ & $0.987\pm0.01$  & \\
Loan & $0.795\pm0.05$ & $

In [ ]:
for bias_type in bias_types:
    print(f"{bias_type}, {0.1}")
    for dataset in data_set_replacer.keys():
        mean_mcc_values = []
        std_mcc_values = []
        for method in result_df["Method"].unique():
            try:
                mean_mcc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==0.1) & 
                                                    (result_df["Data Set"]==dataset)]["MCC Mean"].iloc[0]
                mean_mcc_values.append(np.round(mean_mcc, 3))

                std_mcc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==0.1) & 
                                                    (result_df["Data Set"]==dataset)]["MCC Std"].iloc[0]
                std_mcc_values.append(np.round(std_mcc, 2))
            except IndexError:
                mean_mcc_values.append(0)
                std_mcc_values.append(0)

        print(f"{data_set_replacer[dataset]} \
& ${mean_mcc_values[0]}\\pm{std_mcc_values[0]}$ \
& ${mean_mcc_values[1]}\\pm{std_mcc_values[1]}$ \
& ${mean_mcc_values[2]}\\pm{std_mcc_values[2]}$ \
& ${mean_mcc_values[3]}\\pm{std_mcc_values[3]}$ \
& ${mean_mcc_values[4]}\\pm{std_mcc_values[4]}$ \
& ${mean_mcc_values[5]}\\pm{std_mcc_values[5]}$ \
& ${mean_mcc_values[6]}\\pm{std_mcc_values[6]}$  \
& \\\\")
print("\n")

In [20]:
result_df.round(3)

,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std
0,Uniform,diabetes,0.791,0.022,0.371,0.038,less_positive_class,0.1,0.00,0.000
1,KMM,diabetes,0.781,0.022,0.357,0.042,less_positive_class,0.1,0.00,0.000
2,PSA,diabetes,0.787,0.023,0.367,0.037,less_positive_class,0.1,0.00,0.000
3,Soft-MRS,diabetes,0.788,0.022,0.368,0.038,less_positive_class,0.1,0.00,0.000
4,MRS,diabetes,0.789,0.023,0.367,0.040,less_positive_class,0.1,49.90,32.117
5,FW-MRS,diabetes,0.784,0.024,0.360,0.039,less_positive_class,0.1,45.20,33.882
6,FW-MRS$_{SVM}$,diabetes,0.782,0.030,0.356,0.047,less_positive_class,0.1,82.70,34.441
7,Uniform,folktables_employment,0.871,0.010,0.827,0.017,less_positive_class,0.1,0.00,0.000
8,KMM,folktables_employment,0.857,0.014,0.809,0.022,less_positive_class,0.1,0.00,0.000
9,PSA,folktables_employment,0.867,0.011,0.824,0.017,less_positive_class,0.1,0.04,0.280


In [ ]:
result_df["Rank AUROC"] = result_df.round(3).groupby("Data Set")["AUROC Mean"].rank(ascending=False)
result_df["Rank AUPRC"] = result_df.round(3).groupby("Data Set")["AUPRC Mean"].rank(ascending=False)
result_df["Rank MCC"] = result_df.round(3).groupby("Data Set")["MCC Mean"].rank(ascending=False)
result_df[["Method", "Rank AUROC", "Rank AUPRC", "Rank MCC"]].groupby("Method").mean()

,Rank AUROC,Rank AUPRC
Method,,
FW-MRS,4.5625,4.7500
FW-MRS$_{SVM}$,5.3750,5.5625
KMM,6.0000,5.8125
MRS,2.0000,1.7500
PSA,4.2500,3.8750
Soft-MRS,4.2500,4.5000
Uniform,1.5625,1.7500
